In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

pd.set_option('display.max_columns', None)

In [3]:
#design parameters for the first iteration of the feature engineering process
MC = 2.0 #magnitude of completeness
BIN_WIDTH = 0.1 #USGS catalog magnitude rounding

SEISMICITY_WINDOWS = {7, 30, 90} #time windows for seismicity analysis in days
B_VALUE_WINDOW = 90 #trailing window for b-value calculation
B_VALUE_MIN_EVENTS = 10 #minimum number of events required to calculate a b-value
STRAIN_WINDOW = 90 #trailing window for strain calculation
STRAIN_MAX_MISSING_FRAC = 0.3 #skip window if >30% of strain data is missing
STRAIN_MIN_VALID_DAYS = 20 #minimum number of valid days required to calculate strain

MAX_ASSIGN_DIST_KM = 20 #earthquakes further than this distance from any segment are dropped
N_NEAREST_STATIONS = 3 #number of nearest stations to use for strain calculation

PROJECTED_CRS = "EPSG:3310" #projected coordinate reference system for distance calculations since lat/lon is not a Euclidean space


In [ ]:
#loading all the files in the data folder
quakes = pd.read_parquet("../data/raw/usgs_2016_2026.parquet")
gps = pd.read_parquet("../data/raw/gps_raw.parquet")
faults = gpd.read_file("../data/raw/san_andreas_faults.geojson")

quakes["time"] = pd.to_datetime(quakes["time"]).dt.tz_localize(None)
gps["date"] = pd.to_datetime(gps["date"])

print(f"Loaded {len(quakes)} earthquakes, {len(gps)} GPS stations across {gps['station_code'].nunique()} unique stations, and {len(faults)} fault segments.")

faults.head()

Loaded 115828 earthquakes, 68378 GPS stations across 8 unique stations, and 8042 fault segments.


,fault_name,section_na,fault_id,section_id,Location,linetype,age,dip_direct,slip_rate,slip_sense,scale,class,certainty,strike,fault_leng,cooperator,earthquake,review_dat,fault_url,symbology,ref_id,Shape_Leng,geometry
0,San Andreas fault zone,Shelter Cove Section,1,a,California,Inferred,historic,Vertical,Greater than 5.0 mm/yr,Right lateral,unspecified,A,Good,N12°W,1082,California Geological Survey,San Francisco earthquake,2002-12-10,https://earthquake.usgs.gov/cfusion/qfault/sho...,historic Inferred,1a,8505.454600,"LINESTRING (-124.07161 40.06145, -124.08192 40..."
1,San Andreas fault zone,Shelter Cove Section,1,a,California,Inferred,historic,Vertical,Greater than 5.0 mm/yr,Right lateral,unspecified,A,Good,N12°W,1082,California Geological Survey,San Francisco earthquake,2002-12-10,https://earthquake.usgs.gov/cfusion/qfault/sho...,historic Inferred,1a,312.395544,"LINESTRING (-124.078 40.04641, -124.07949 40.0..."
2,San Andreas fault zone,Shelter Cove Section,1,a,California,Inferred,historic,Vertical,Greater than 5.0 mm/yr,Right lateral,unspecified,A,Good,N12°W,1082,California Geological Survey,San Francisco earthquake,2002-12-10,https://earthquake.usgs.gov/cfusion/qfault/sho...,historic Inferred,1a,389.442401,"LINESTRING (-124.07643 40.04447, -124.07463 40..."
3,San Andreas fault zone,Shelter Cove Section,1,a,California,Inferred,historic,Vertical,Greater than 5.0 mm/yr,Right lateral,unspecified,A,Good,N12°W,1082,California Geological Survey,San Francisco earthquake,2002-12-10,https://earthquake.usgs.gov/cfusion/qfault/sho...,historic Inferred,1a,370.169121,"LINESTRING (-124.07384 40.0412, -124.07281 40...."
4,San Andreas fault zone,Shelter Cove Section,1,a,California,Moderately Constrained,historic,Vertical,Greater than 5.0 mm/yr,Right lateral,unspecified,A,Good,N12°W,1082,California Geological Survey,San Francisco earthquake,2002-12-10,https://earthquake.usgs.gov/cfusion/qfault/sho...,historic Moderately Constrained,1a,79.867780,"LINESTRING (-124.07747 40.04578, -124.07787 40..."


### Spatial Join - assigning the earthquakes to fault segments ###

In [29]:
#assigning stable segment ID before any projection happens
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

print(F"Fault sections: {len(faults)}")

#building a GeoDataFrame for the earthquakes and projecting to a projected coordinate reference system for distance calculations
quake_points = gpd.GeoDataFrame(
    quakes.copy(),
    geometry = gpd.points_from_xy(quakes["longitude"], quakes["latitude"]),
    crs = "EPSG:4326"
)

quake_points_m = quake_points.to_crs(PROJECTED_CRS)
faults_m = faults.to_crs(PROJECTED_CRS)

max_dist_m = MAX_ASSIGN_DIST_KM * 1000 #convert to meters for distance calculations

joined = gpd.sjoin_nearest(
    quake_points_m,
    faults_m[["fault_segment_id", "geometry"]],
    max_distance = max_dist_m,
    distance_col = "dist_to_fault_m"
)

n_dropped = len(quake_points_m) - len(joined)
print(f"{len(joined)} earthquakes assigned to a fault segment, {n_dropped} dropped due to distance > {MAX_ASSIGN_DIST_KM} km.")

quakes_assigned = joined.drop(columns = "geometry").rename(columns = {"time": "event_time"})


quakes_assigned["dist_to_fault_km"] = quakes_assigned["dist_to_fault_m"] / 1000
quakes_assigned[["event_time", "magnitude", "fault_segment_id", "dist_to_fault_km"]].head()

Fault sections: 10
12030 earthquakes assigned to a fault segment, 103798 dropped due to distance > 20 km.


,event_time,magnitude,fault_segment_id,dist_to_fault_km
35,2016-01-02 07:50:35.200000+00:00,1.81,7,9.556607
45,2016-01-02 18:02:32.490000+00:00,1.52,7,10.854013
49,2016-01-03 01:49:09.860000+00:00,2.10,2,1.172243
59,2016-01-03 10:06:13.350000+00:00,1.60,0,2.319853
66,2016-01-03 15:59:18.200000+00:00,1.81,2,0.173774


In [28]:
print(faults.columns.tolist())          # does it have a fault-name field we could have filtered on?
print(faults.total_bounds)              # [minx, miny, maxx, maxy] - the spatial extent
print(faults["fault_name"].nunique())
print(faults["fault_name"].dropna().unique()[:20])
print(faults["section_na"].nunique())
print(faults["section_na"].dropna().unique())
print(faults["section_na"].isna().sum())
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

print(len(faults))
print(faults[["section_na", "fault_segment_id"]])

['fault_name', 'section_na', 'fault_id', 'section_id', 'Location', 'linetype', 'age', 'dip_direct', 'slip_rate', 'slip_sense', 'scale', 'class', 'certainty', 'strike', 'fault_leng', 'cooperator', 'earthquake', 'review_dat', 'fault_url', 'symbology', 'ref_id', 'Shape_Leng', 'geometry', 'fault_segment_id']
[-124.090694   33.351332 -115.715781   40.117564]
1
['San Andreas fault zone']
10
['Shelter Cove Section' 'North Coast section' 'Peninsula section'
 'Santa Cruz Mountains section' 'Creeping section' 'Parkfield section'
 'Cholame-Carrizo section' 'Mojave section'
 'San Bernardino Mountains section' 'Coachella section']
0
10
                         section_na fault_segment_id
0           Cholame-Carrizo section                0
1                 Coachella section                1
2                  Creeping section                2
3                    Mojave section                3
4               North Coast section                4
5                 Parkfield section                

### Seismicity Rate - rolling event counts per segment ###

In [30]:
def build_daily_segment_counts(df, segment_col = "fault_segment_id", time_col = "event_time"):
    """ one row per segment per day, with counts of earthquakes in each magnitude bin """

    df = df.copy()
    df["date"] = df[time_col].dt.floor("D")
    daily = (
      df.groupby([segment_col, "date"])
      .size()
      .reset_index(name = "event_count")
    )

    filled_frames = []
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq = "D")
    for seg_id, sub in daily.groupby(segment_col):
        sub = sub.set_index("date").reindex(full_range, fill_value = 0)
        sub[segment_col] = seg_id
        sub = sub.rename_axis("date").reset_index()
        filled_frames.append(sub)
    return pd.concat(filled_frames, ignore_index = True) 

daily_counts = build_daily_segment_counts(quakes_assigned)

seismicity = daily_counts.sort_values(["fault_segment_id", "date"]).copy()
for w in SEISMICITY_WINDOWS:
    seismicity[f"seismicity_rate_{w}d"] = (
        seismicity.groupby("fault_segment_id")["event_count"]
        .transform(lambda s: s.rolling(window = w, min_periods = 1).sum())
    )

def zscore_by_segment(group, col):
    """ z-score a column within each fault segment """
    mu, sigma = group[col].mean(), group[col].std()
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(0, index = group.index)
    else:
        return (group[col] - mu) / sigma

seismicity["seismicity_rate_30d_z"] = (
    seismicity.groupby("fault_segment_id", group_keys = False)
    .apply(lambda g: zscore_by_segment(g, "seismicity_rate_30d"))
)

seismicity.drop(columns = "event_count").head()

C:\Users\matth\AppData\Local\Temp\ipykernel_24928\2638084305.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: zscore_by_segment(g, "seismicity_rate_30d"))


,date,fault_segment_id,seismicity_rate_90d,seismicity_rate_30d,seismicity_rate_7d,seismicity_rate_30d_z
0,2016-01-02 00:00:00+00:00,0,0.0,0.0,0.0,-1.507617
1,2016-01-03 00:00:00+00:00,0,1.0,1.0,1.0,-1.067257
2,2016-01-04 00:00:00+00:00,0,1.0,1.0,1.0,-1.067257
3,2016-01-05 00:00:00+00:00,0,1.0,1.0,1.0,-1.067257
4,2016-01-06 00:00:00+00:00,0,1.0,1.0,1.0,-1.067257


### B-Value - rolling Aki MLE per segment ###

In [32]:
def aki_b_value(magnitudes, mc = MC, bin_width = BIN_WIDTH):
    """ calculate the b-value using the Aki method """
    m = np.array(magnitudes)
    m = m[m >= mc]
    n_events = len(m)
    if n_events < B_VALUE_MIN_EVENTS:
        return np.nan, np.nan, n_events

    mc_shifted = mc - bin_width / 2.0
    mean_m = m.mean()
    b = (np.log10(np.e)) / (mean_m - mc_shifted)
    std_err = 2.30 * b**2 * np.sqrt(
        np.sum((m - mean_m)**2) / (n_events * (n_events - 1))
    )
    return b, std_err, n_events

def rolling_b_value_for_segment(events, dates, window_days = B_VALUE_WINDOW):
    """ calculate the rolling b-value for a single fault segment """
    events = events.sort_values("event_time")
    magnitude = events["magnitude"].values

    results = []
    window = pd.Timedelta(days = window_days)
    for d in dates:
        mask = (events["event_time"] > d - window) & (events["event_time"] <= d)
        b, se, n = aki_b_value(magnitude[mask.values])
        results.append({"date": d, "b_value": b, "b_value_se": se, "n_events": n})
    return pd.DataFrame(results)

date_range = pd.date_range(quakes_assigned["event_time"].min().floor('D'), quakes_assigned["event_time"].max().floor('D'), freq = "D")

b_value_frames = []
for seg_id, seg_events in quakes_assigned.groupby("fault_segment_id"):
    b_values_seg = rolling_b_value_for_segment(seg_events, date_range)
    b_values_seg["fault_segment_id"] = seg_id
    b_value_frames.append(b_values_seg)

b_values = pd.concat(b_value_frames, ignore_index = True)
print(b_values.isna().mean())
print(b_values["n_events"].describe())
b_values[b_values["n_events"] >= B_VALUE_MIN_EVENTS].head()

'''print(len(faults))                                    # how many segments did we actually load?
print(quakes_assigned["fault_segment_id"].nunique())  # how many distinct segments got ANY earthquake at all?
print(quakes_assigned.groupby("fault_segment_id").size().describe())  # total events per segment, over the whole catalog'''

date                0.000000
b_value             0.715389
b_value_se          0.715389
n_events            0.000000
fault_segment_id    0.000000
dtype: float64
count    36520.000000
mean        10.352547
std         16.430837
min          0.000000
25%          1.000000
50%          3.000000
75%         12.000000
max        122.000000
Name: n_events, dtype: float64


'print(len(faults))                                    # how many segments did we actually load?\nprint(quakes_assigned["fault_segment_id"].nunique())  # how many distinct segments got ANY earthquake at all?\nprint(quakes_assigned.groupby("fault_segment_id").size().describe())  # total events per segment, over the whole catalog'

In [34]:
per_segment = b_values.merge(
    faults[["fault_segment_id", "section_na"]], on="fault_segment_id"
)
nan_by_section = per_segment.groupby("section_na")["b_value"].apply(lambda s: s.isna().mean())
print(nan_by_section.sort_values(ascending=False))

lifetime_counts = quakes_assigned.groupby("fault_segment_id").size()
print(lifetime_counts.rename("total_events").sort_values())

section_na
North Coast section                 1.000000
Mojave section                      1.000000
Parkfield section                   1.000000
Shelter Cove Section                1.000000
Cholame-Carrizo section             0.992059
Peninsula section                   0.933461
Coachella section                   0.806134
Santa Cruz Mountains section        0.399507
San Bernardino Mountains section    0.015060
Creeping section                    0.007667
Name: b_value, dtype: float64
fault_segment_id
5     140
9     146
4     205
3     271
0     420
6     562
8    1185
1    1468
7    3037
2    4596
Name: total_events, dtype: int64


### GPS Strain Rate - gap-aware rolling slope, then distance weighted to segments ###

In [41]:
def gap_aware_slope(window_series, max_missing_frac=STRAIN_MAX_MISSING_FRAC,
                     min_valid_days=STRAIN_MIN_VALID_DAYS):
    """Linear regression slope (per year) over a window, NaN if too much is missing."""
    valid = window_series.dropna()
    missing_frac = 1 - len(valid) / len(window_series)
    if missing_frac > max_missing_frac or len(valid) < min_valid_days:
        return np.nan
    x = (valid.index - valid.index[0]).days.values.astype(float)
    y = valid.values
    slope, _ = np.polyfit(x, y, 1)
    return slope * 365  # daily slope -> mm/year


def rolling_station_strain(gps_df, window_days=STRAIN_WINDOW, start_date=None, end_date=None):
    """Per-station rolling strain-rate slope for north and east components."""
    frames = []
    for station, sub in gps_df.groupby("station_code"):
        sub = sub.set_index("date").sort_index()

        range_start = (start_date - pd.Timedelta(days=window_days)) if start_date else sub.index.min()
        range_end = end_date if end_date else sub.index.max()
        full_idx = pd.date_range(range_start, range_end, freq="D")
        sub = sub.reindex(full_idx)  # explicit NaNs on missing days, incl. the CRFP-style gap
        
        results = []
        window = pd.Timedelta(days=window_days)
        for d in full_idx:
            win = sub.loc[d - window: d]
            n_slope = gap_aware_slope(win["north_mm"])
            e_slope = gap_aware_slope(win["east_mm"])
            results.append((d, station, n_slope, e_slope))
        frames.append(pd.DataFrame(results, columns=["date", "station_code",
                                                       "north_strain_rate", "east_strain_rate"]))
    return pd.concat(frames, ignore_index=True)

station_strain = rolling_station_strain(
    gps,
    start_date = quakes_assigned["event_time"].min().floor('D'),
    end_date = quakes_assigned["event_time"].max().floor('D')
)
station_strain.head()

print(station_strain[["north_strain_rate", "east_strain_rate"]].isna().mean())
print(station_strain.groupby("station_code")[["north_strain_rate", "east_strain_rate"]].apply(lambda g: g.isna().mean()))


north_strain_rate    1.0
east_strain_rate     1.0
dtype: float64
              north_strain_rate  east_strain_rate
station_code                                     
BKMS                        1.0               1.0
CMBB                        1.0               1.0
COPR                        1.0               1.0
CRFP                        1.0               1.0
HOPB                        1.0               1.0
P066                        1.0               1.0
P495                        1.0               1.0
P506                        1.0               1.0


datetime64[ns]
datetime64[ns]


In [37]:
crfp = gps[gps["station_code"] == "CRFP"].set_index("date").sort_index()
full_idx = pd.date_range(crfp.index.min(), crfp.index.max(), freq="D")
crfp = crfp.reindex(full_idx)

missing = crfp["north_mm"].isna()
gap_lengths = missing.groupby((~missing).cumsum()).sum()
print("Largest contiguous gap (days):", gap_lengths.max())
print("Total missing days:", missing.sum(), "/", len(crfp))

Largest contiguous gap (days): 485
Total missing days: 761 / 11030


In [49]:
print(gps[gps["station_code"] == "HOPB"]["station_name"].unique())
print(gps[gps["station_code"] == "P506"]["station_name"].unique())
print(gps[gps["station_name"] == "Indio (S. San Andreas)"]["station_code"].unique())


['Hopland']
['Indio (S. San Andreas)']
['P506']


In [53]:
#getting station coords and dealing with distance weighting for strain rates
station_coords = station_coords = pd.DataFrame({
    "station_code": ["BKMS", "CMBB", "COPR", "CRFP", "HOPB", "P066", "P495", "P506"],
    "lat": [33.962, 38.034, 34.415, 34.039, 38.995, 32.617, 33.045, 33.768],
    "lon": [-118.095, -120.386, -119.880, -117.100, -123.075, -116.170, -115.628, -116.239],
})

missing = set(gps["station_code"].unique()) - set(station_coords["station_code"])
print("Codes in gps but missing from station_coords:", missing)

def inverse_distance_weight_to_segments(faults_m, station_points_m, values_df, n_nearest=N_NEAREST_STATIONS):
    """Values_df should have columns: station_code, date, north_strain_rate, east_strain_rate,
    returns per segment per day strain rates by inverse distance weighting from the nearest stations."""
    seg_centroids = faults_m.copy()
    seg_centroids["geometry"] = seg_centroids.geometry.centroid

    # Distance matrix: segment centroid -> every station
    dists = seg_centroids.geometry.apply(lambda pt: station_points_m.geometry.distance(pt))
    dists.index = seg_centroids["fault_segment_id"]
    dists.columns = station_points_m["station_code"]

    records = []
    for seg_id, row in dists.iterrows():
        nearest = row.nsmallest(n_nearest)
        weights = 1 / nearest.replace(0,1)
        weights = weights / weights.sum()  # normalize to sum to 1
        records.append({
            "fault_segment_id": seg_id,
            "nearest_stations": list(nearest.index),
            "weights": weights.values,
            "gps_station_distance_km": nearest.min() / 1000
        })
    weight_table = pd.DataFrame(records)

    merged = []
    for date, day_df in values_df.groupby("date"):
        day_lookup = day_df.set_index("station_code")
        for _, wrow in weight_table.iterrows():
            stations = wrow["nearest_stations"]
            w = wrow["weights"]
            present = [s for s in stations if s in day_lookup.index]
            if not present:
                continue
            n_vals = day_lookup.loc[present, "north_strain_rate"].values
            e_vals = day_lookup.loc[present, "east_strain_rate"].values
            w_present = w[[stations.index(s) for s in present]]
            w_present = w_present / w_present.sum()  # renormalize
            merged.append({
                "fault_segment_id": wrow["fault_segment_id"],
                "date": date,
                "gps_strain_rate": np.nansum(np.hypot(n_vals, e_vals) * w_present),
                "gps_station_distance_km": wrow["gps_station_distance_km"]
            })
    return pd.DataFrame(merged)

station_points_gdf = gpd.GeoDataFrame(
    station_coords,
    geometry=gpd.points_from_xy(station_coords["lon"], station_coords["lat"]),
    crs="EPSG:4326",
).to_crs(PROJECTED_CRS)

segment_strain = inverse_distance_weight_to_segments(faults_m, station_points_gdf, station_strain)
segment_strain.head()
print(segment_strain["gps_station_distance_km"].describe())


Codes in gps but missing from station_coords: set()
count    37420.000000
mean       111.307947
std         62.528424
min         26.880825
25%         54.430412
50%        115.708449
75%        169.250580
max        192.051904
Name: gps_station_distance_km, dtype: float64


In [54]:
segment_strain_with_names = segment_strain.merge(
    faults[["fault_segment_id", "section_na"]], on="fault_segment_id"
)
print(segment_strain_with_names.groupby("section_na")["gps_station_distance_km"].mean().sort_values())

section_na
Coachella section                    26.880825
San Bernardino Mountains section     27.691554
North Coast section                  54.430412
Mojave section                       64.776567
Cholame-Carrizo section              84.929555
Shelter Cove Section                146.487343
Santa Cruz Mountains section        166.842693
Parkfield section                   169.250580
Peninsula section                   179.738040
Creeping section                    192.051904
Name: gps_station_distance_km, dtype: float64


In [56]:
GPS_RELIABLE_MAX_KM = 100  # design choice: segments farther than this from their nearest
                            # weighted GPS stations are flagged low-confidence.
                            # Given your real distances ranged 27-192 km, this splits
                            # roughly in half -- Coachella/San Bernardino/North Coast/
                            # Mojave/Cholame-Carrizo land reliable, the rest don't.
                            # Worth revisiting once you see how it affects model performance.

b_values["b_value_reliable"] = b_values["n_events"] >= B_VALUE_MIN_EVENTS
segment_strain["gps_reliable"] = segment_strain["gps_station_distance_km"] <= GPS_RELIABLE_MAX_KM

features = seismicity.merge(b_values, on=["fault_segment_id", "date"], how="left")
features = features.merge(segment_strain, on=["fault_segment_id", "date"], how="left")
features = features.sort_values(["fault_segment_id", "date"]).reset_index(drop=True)

print(features.shape)
print(features.columns.tolist())

print(features[["b_value_reliable", "gps_reliable"]].isna().sum())
print(features["fault_segment_id"].nunique())   # should be 10

(36520, 14)
['date', 'fault_segment_id', 'event_count', 'seismicity_rate_90d', 'seismicity_rate_30d', 'seismicity_rate_7d', 'seismicity_rate_30d_z', 'b_value', 'b_value_se', 'n_events', 'b_value_reliable', 'gps_strain_rate', 'gps_station_distance_km', 'gps_reliable']
b_value_reliable    0
gps_reliable        0
dtype: int64
10


In [57]:
import os
os.makedirs("../data/processed", exist_ok=True)
features.to_parquet("../data/processed/features.parquet", index=False)
print("Saved data/processed/features.parquet")

Saved data/processed/features.parquet
